# 131 — Contratos de roles, capacidades y resultados

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) **válida** — 0.0 está dentro de [0,1] y "sin secretos" no es
vacía. (b) violación de **tipo**: score es string "0.6", no number (el error más
común al parsear LLM). (c) violación de **minLength**: finding vacía. (d) violación
de **const**: "seguridad" ≠ "security" — los enums en español/inglés mezclados son un
clásico.

**Ejercicio 2.** El validador acumula *todas* las violaciones (no corta en la
primera): el feedback completo hace el reintento más eficaz.

**Ejercicio 3.** La clave es que el error final es *tipado y honesto*: quien consume
sabe que hubo k intentos y por qué fallaron, en lugar de recibir un valor "arreglado".

**Ejercicio 4.** Los 3 workers pasan (const parametrizado por nombre). El contrato de
resultados del sistema completo es el JSON raíz: `kind`, `seed`, `result`,
`evidence`, `limitations` — la misma idea a otra escala: cualquier consumidor del
laboratorio valida esas claves antes de usarlo.


In [ ]:
result = run_lab("multiagent", seed=131)
assert result["kind"] == "multiagent"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 2
def valida(out, agente="security"):
    v = []
    if not isinstance(out, dict):
        return False, ["tipo: la salida no es un objeto"]
    if out.get("agent") != agente:
        v.append(f"const: agent != '{agente}'")
    score = out.get("score")
    if not isinstance(score, (int, float)) or isinstance(score, bool):
        v.append("tipo: score no es number")
    elif not 0 <= score <= 1:
        v.append("rango: score fuera de [0,1]")
    finding = out.get("finding")
    if not isinstance(finding, str):
        v.append("tipo/requerido: finding ausente o no string")
    elif len(finding) < 1:
        v.append("minLength: finding vacía")
    return (not v), v

casos = [
    {"agent": "security", "score": 0.0, "finding": "sin secretos"},
    {"agent": "security", "score": "0.6", "finding": "ok"},
    {"agent": "security", "score": 0.6, "finding": ""},
    {"agent": "seguridad", "score": 0.6, "finding": "ok"},
]
for c in casos:
    print(valida(c))

# Ejercicio 3
def invocar_con_contrato(fn, k=3, agente="security"):
    feedback = []
    for intento in range(1, k + 1):
        out = fn(intento, feedback)
        ok, feedback = valida(out, agente)
        if ok:
            return out
    return {"error": "contract_violation", "attempts": k, "last": feedback}

def worker_flaky(intento, feedback):
    if intento < 3:
        return {"agent": "security", "score": 1.4, "finding": "sobreconfianza"}
    return {"agent": "security", "score": 0.6, "finding": "falta threat model"}

print(invocar_con_contrato(worker_flaky))

# Ejercicio 4
result = run_lab("multiagent", seed=131)
for w in result["result"]["workers"]:
    ok, viols = valida(w, agente=w["agent"])
    assert ok, viols
assert {"kind", "seed", "result", "evidence", "limitations"} <= set(result)
print("3 workers válidos; contrato global:", sorted(result.keys()))


## Reflexión

1. El laboratorio devuelve siempre resultados válidos porque los workers son deterministas. ¿En qué punto exacto del flujo insertarías el validador si fueran LLM, y qué harías con la evidencia de los intentos fallidos?
2. ¿Por qué "score 1.4 → recortar a 1.0 en silencio" es peor que "score 1.4 → reintento visible", si el segundo cuesta una llamada extra?
3. Diseña una garantía de SLA de *calidad* para el worker documentation que puedas medir de verdad esta semana. ¿Qué muestra, qué juez, qué umbral?
